## Purpose: 

- Test Docling table extraction strategies against Python spatial extraction functions

In [1]:
## Imports

import os

if os.name == "nt":
    os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
    print("Windows local workaround active: KMP_DUPLICATE_LIB_OK=TRUE")

Windows local workaround active: KMP_DUPLICATE_LIB_OK=TRUE


In [2]:
import torch

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional


from tqdm.notebook import tqdm

from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

from sentence_transformers import SentenceTransformer
from unstructured.partition.pdf import partition_pdf
import pdfplumber


import pickle


### **Section One:** Extract table from Antam Sample Page using spatial extraction methods

In [6]:
## Function definitions for spatial extraction 

def infer_label_boundary(words: list[dict], gap_threshold: float = 30.0) -> float:
    """Infer the x-coordinate boundary between label and value columns.

    Finds the largest horizontal gap in word x-positions, which typically
    separates the row label column from the numeric value columns.

    Args:
        words: List of word dicts from pdfplumber's extract_words().
        gap_threshold: Minimum gap size to consider as a column boundary.

    Returns:
        x-coordinate of the inferred boundary, or a fallback of half page width.
    """
    if not words:
        return 270.0

    x_positions = sorted(set(round(w["x0"]) for w in words))
    if len(x_positions) < 2:
        return 270.0

    gaps = [
        (x_positions[i + 1] - x_positions[i], x_positions[i])
        for i in range(len(x_positions) - 1)
    ]
    large_gaps = [(gap, x) for gap, x in gaps if gap >= gap_threshold]

    if not large_gaps:
        return (x_positions[0] + x_positions[-1]) / 2

    largest_gap = max(large_gaps, key=lambda g: g[0])
    return float(largest_gap[1] + largest_gap[0] / 2)


def cluster_into_rows(
    words: list[dict],
    tolerance: int = 3,
) -> dict[float, list[tuple]]:
    """Group pdfplumber word dicts into rows by snapping y0 to a grid.

    Args:
        words: List of word dicts from pdfplumber's extract_words().
        tolerance: Pixel tolerance for grouping words into the same row.

    Returns:
        Dict mapping snapped y-coordinate to list of (x0, text) tuples.
    """
    rows: dict[float, list[tuple]] = {}
    for word in words:
        y_key = round(word["top"] / tolerance) * tolerance
        if y_key not in rows:
            rows[y_key] = []
        rows[y_key].append((word["x0"], word["text"]))
    return rows


def rows_to_lines(
    rows: dict[float, list[tuple]],
    label_boundary: float = 270.0,
    min_words_per_row: int = 2,
) -> list[str]:
    """Join label words as prose, separate value columns with |.

    Args:
        rows: Output of cluster_into_rows().
        label_boundary: x-coordinate threshold separating labels from values.
        min_words_per_row: Minimum words required to include a row.

    Returns:
        List of strings, one per row.
    """
    result = []
    for y_key in sorted(rows.keys()):
        row_words = sorted(rows[y_key], key=lambda w: w[0])

        if len(row_words) < min_words_per_row:
            continue

        labels = [w[1] for w in row_words if w[0] < label_boundary]
        values = [w[1] for w in row_words if w[0] >= label_boundary]

        parts = []
        if labels:
            parts.append(" ".join(labels))
        if values:
            parts.append(" | ".join(values))

        if parts:
            result.append(" | ".join(parts))

    return result


def extract_table_page_spatial(
    page: pdfplumber.page.Page,
    gap_threshold: float = 30.0,
    tolerance: int = 3,
    min_words_per_row: int = 2,
) -> str:
    """Extract a table page using spatial clustering.

    Args:
        page: A pdfplumber Page object.
        gap_threshold: Minimum gap for inferring label boundary.
        tolerance: Pixel tolerance for row clustering.
        min_words_per_row: Minimum words required to include a row.

    Returns:
        Extracted text as a single string.
    """
    words = page.extract_words()
    label_boundary = infer_label_boundary(words, gap_threshold=gap_threshold)
    rows = cluster_into_rows(words, tolerance=tolerance)
    lines = rows_to_lines(
        rows,
        label_boundary=label_boundary,
        min_words_per_row=min_words_per_row,
    )
    return "\n".join(lines)

In [ ]:
# create spatial_extraction.md

pdf_path = Path(r"C:\Users\elimo\problem-set-2-elimossmarks11\data\interim\table_extraction_comparison\antam_sample_table.pdf")
output_path = Path(r'C:\Users\elimo\problem-set-2-elimossmarks11\data\interim\table_extraction_comparison\spatial_extraction.md')

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    lines = extract_table_page_spatial(page)
    output_path.write_text(lines, encoding="utf-8")

### **Section 3:** Extract Same Table Using Docling

In [11]:
import os
from pathlib import Path

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HUGGINGFACE_HUB_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"  # this is the key one

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

result = converter.convert(
    r"C:\Users\elimo\problem-set-2-elimossmarks11\data\interim\table_extraction_comparison\antam_sample_table.pdf"
)
output_path = Path(r'C:\Users\elimo\problem-set-2-elimossmarks11\data\interim\table_extraction_comparison\docling_extraction.md')
output_path.write_text(result.document.export_to_markdown(), encoding="utf-8")
print(result.document.export_to_markdown())

## Ikhtisar Keuangan

## Financial Highlights

| Deskripsi Description                                                                                                                                                                                     | 2012      | 2013*     | 2014**    | 2015**     | 2016***    |
|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|-----------|-----------|-----------|------------|------------|
| Penjualan Bersih &#124; Net Sales                                                                                                                                                                         | 10.449,88 | 11.298,32 | 9.420,63  | 10.531,50  | 9.106,26   |
| Beban Pokok Penjualan Cost of Goods Sold                                                                                                           